<a href="https://colab.research.google.com/github/Imam-2101/numeric_computation_2026/blob/main/Penyelesaian_Metode_Bisection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mencari Akar Persamaan dengan Metode Bagidua (Bisection) 🔍

**Nama:** Imam Bagus Saputra | **NIM:** 202511420024 | **Prodi:** Teknik Informatika

## Apa yang dicari?

Kita punya sebuah fungsi **f(x) = x³ − 2x − 5**. Kita ingin mencari nilai **x** yang membuat f(x) = 0.
Nilai x itu disebut **akar**. Di grafik, akar adalah titik tempat kurva **memotong sumbu-X**.

## Ide metode bagidua (seperti permainan tebak angka)

Bayangkan teman kita menyimpan angka di antara 2,0 dan 2,1. Kita menebak angka di **tengahnya**,
lalu ia menjawab "lebih besar" atau "lebih kecil". Dengan jawaban itu, **separuh daerah tebakan dibuang**.
Kita ulangi terus sampai daerah tebakannya sangat sempit. Itulah akarnya.

Di metode ini, "jawaban teman" diganti **tanda** f(x): positif (+) atau negatif (−).
Jika f(a) dan f(c) berbeda tanda, akar ada di antara a dan c.

## Cara memakai notebook ini

1. Jalankan cell kode **dari atas ke bawah** (tekan ▶ atau Shift + Enter).
2. Jangan loncat, karena cell di bawah memakai hasil cell di atas.
3. Untuk mencoba soal lain, cukup ubah **cell kode pertama** (Langkah 1), lalu jalankan ulang semuanya (**Runtime > Run all**).

## Alur kerja

1. Pengaturan dan fungsi f(x)
2. Fungsi pencari selang (dari tabel nilai)
3. Fungsi bagidua (inti metode)
4. Fungsi pembuat kurva
5. Mencari selang yang mengandung akar
6. **Diagram 1**: grafik f(x) dan selang berakar
7. Menjalankan metode bagidua
8. **Diagram 2**: proses pembagian selang
9. Hasil akhir

## Langkah 1: Pengaturan dan Fungsi f(x) ⚙️

Cell berikutnya berisi **semua angka yang boleh Anda ubah**.

| Nama | Artinya | Nilai di sini |
|---|---|---|
| `f(x)` | Fungsi yang dicari akarnya | x³ − 2x − 5 |
| `A_AWAL`, `B_AKHIR` | Rentang x yang kita periksa | 1 sampai 3 |
| `H` | Jarak antar titik yang diperiksa | 0,1 |
| `EPSILON1` | Berhenti jika **lebar selang** lebih kecil dari ini | 0,00001 |
| `EPSILON2` | Berhenti jika **\|f(c)\|** lebih kecil dari ini (hampir nol) | 0,00000001 |
| `DELTA` | Berhenti jika **galat relatif** lebih kecil dari ini | 0,000001 |
| `MAKS_ITER` | Batas jumlah putaran (pengaman) | 100 |
| `LEBAR`, `TINGGI` | Ukuran diagram dalam piksel | 650 dan 340 |

**Tiga kriteria berhenti** (sesuai materi) berarti iterasi berhenti jika **salah satu** terpenuhi.

**Library `plotly`** dipakai untuk membuat grafik interaktif. Library ini sudah tersedia di Colab, jadi tidak perlu dipasang.

### Cara mengganti soal
- Ganti isi `f(x)`, misalnya `return x**2 - 4`.
- Jika memakai `exp`, `sin`, atau `cos`, tambahkan `import math` lalu tulis `math.exp(x)`.
- Ganti `A_AWAL`, `B_AKHIR`, dan `H` agar rentangnya memuat akar.

Cell ini tidak menampilkan output, hanya menyimpan pengaturan.

In [1]:
import plotly.graph_objects as go   # library untuk membuat grafik interaktif


def f(x):
    """Fungsi yang dicari akarnya, f(x) = 0. Ubah di sini untuk soal lain."""
    return x**3 - 2*x - 5


# ---------- Pengaturan (boleh diubah) ----------
A_AWAL, B_AKHIR, H = 1.0, 3.0, 0.1   # rentang pencarian dan kenaikan absis h
EPSILON1 = 1e-5    # kriteria 1: lebar selang baru |a - b| < epsilon1
EPSILON2 = 1e-8    # kriteria 2: |f(c)| < epsilon2 (mendekati nol)
DELTA = 1e-6       # kriteria 3: galat relatif < delta
MAKS_ITER = 100    # batas iterasi (pengaman supaya tidak berputar selamanya)

LEBAR, TINGGI = 650, 340   # ukuran diagram (piksel), kecil tapi tetap jelasimport plotly.graph_objects as go   # library untuk membuat grafik interaktif


def f(x):
    """Fungsi yang dicari akarnya, f(x) = 0. Ubah di sini untuk soal lain."""
    return x**3 - 2*x - 5


# ---------- Pengaturan (boleh diubah) ----------
A_AWAL, B_AKHIR, H = 1.0, 3.0, 0.1   # rentang pencarian dan kenaikan absis h
EPSILON1 = 1e-5    # kriteria 1: lebar selang baru |a - b| < epsilon1
EPSILON2 = 1e-8    # kriteria 2: |f(c)| < epsilon2 (mendekati nol)
DELTA = 1e-6       # kriteria 3: galat relatif < delta
MAKS_ITER = 100    # batas iterasi (pengaman supaya tidak berputar selamanya)

LEBAR, TINGGI = 650, 340   # ukuran diagram (piksel), kecil tapi tetap jelas

## Langkah 2: Fungsi Pencari Selang 🔎

Metode bagidua **tidak bisa mulai** tanpa selang [a, b] yang sudah pasti berisi akar.
Syaratnya: **f(a) dan f(b) harus berbeda tanda**, atau f(a) × f(b) < 0.

Cara menemukannya (sesuai materi): buat **tabel nilai f(x)** dengan jarak tetap h, lalu cari dua titik bertetangga yang tandanya berbeda.

Fungsi `cari_selang` bekerja begini:
1. Hitung banyak langkah: n = (b − a) / h. Untuk soal ini n = (3 − 1) / 0,1 = 20, jadi ada **21 titik**.
2. Buat titik x: 1,0 ; 1,1 ; 1,2 ; ... ; 3,0.
3. Hitung f(x) di setiap titik.
4. Untuk tiap dua titik bertetangga, jika hasil kali f-nya **negatif**, simpan pasangan itu.

**Contoh:** f(1,9) = −1,941 ; f(2,0) = −1,000 ; f(2,1) = +0,061.
Antara 2,0 dan 2,1 tandanya berubah dari − ke +, berarti kurva memotong sumbu-X di sana. Selang **[2,0 ; 2,1]** berisi akar.

Cell ini hanya mendefinisikan fungsi, jadi tidak ada output.

In [2]:
def cari_selang(a, b, h):
    """Menghitung f(x) di titik-titik berjarak h dari a sampai b,
    lalu mencari selang yang nilai f-nya berbeda tanda."""
    n = round((b - a) / h)                      # banyak langkah
    x = [a + i * h for i in range(n + 1)]       # titik-titik x: a, a+h, a+2h, ...
    fx = [f(v) for v in x]                      # nilai f di tiap titik
    selang = [(x[i], x[i + 1])                  # simpan pasangan titik yang
              for i in range(n)                 # hasil kali f-nya negatif
              if fx[i] * fx[i + 1] < 0]
    return x, fx, selang

## Langkah 3: Fungsi Bagidua (Inti Metode) ⚙️

Fungsi `bagidua(a, b)` mengerjakan metode di materi. Ia memakai **satu putaran** yang diulang-ulang. Tiap putaran:

| Urutan | Yang dikerjakan | Rumus |
|---|---|---|
| 1 | Cari titik tengah | c = (a + b) / 2 |
| 2 | Hitung f(a), f(c), f(b) | |
| 3 | Periksa tanda | Apakah f(a) × f(c) < 0 ? |
| 4a | Jika **ya**, akar di kiri | selang baru [a, c], yaitu **b ← c** |
| 4b | Jika **tidak**, akar di kanan | selang baru [c, b], yaitu **a ← c** |
| 5 | Hitung lebar selang baru | \|a − b\| |
| 6 | Cek kriteria berhenti | Berhenti jika salah satu terpenuhi |

### Contoh nyata (soal ini, selang awal [2,0 ; 2,1])

- **r = 0:** c = 2,05 dan f(c) = −0,485. f(a) = −1 sama tanda dengan f(c), jadi akar di kanan: **[c, b]**, lebar 0,05.
- **r = 1:** c = 2,075 dan f(c) = −0,216, sama tanda lagi: **[c, b]**, lebar 0,025.
- **r = 2:** c = 2,0875 dan f(c) = −0,078: **[c, b]**, lebar 0,0125.
- **r = 3:** c = 2,09375 dan f(c) = −0,009: **[c, b]**, lebar 0,00625.
- **r = 4:** c = 2,096875 dan f(c) = **+**0,026. Tanda berubah, jadi akar di kiri: **[a, c]**, lebar 0,003125.

Dari contoh terlihat **lebar selang selalu menjadi setengahnya**.

### Tiga kriteria berhenti
1. Lebar selang baru **< EPSILON1** (selang sudah sangat sempit).
2. **|f(c)| < EPSILON2** (nilai fungsi hampir 0).
3. Galat relatif **|c_baru − c_lama| / |c_baru| < DELTA** (c hampir tidak berubah).

### Syarat awal
Fungsi langsung berhenti dengan pesan "Selang tidak valid" jika f(a) × f(b) ≥ 0.

Cell ini hanya mendefinisikan fungsi, jadi tidak ada output.

In [3]:
def bagidua(a, b):
    """Metode bagidua: mencari akar f(x)=0 di dalam selang [a, b].
    Mengembalikan daftar data tiap iterasi."""

    # Syarat cukup keberadaan akar: f(a) * f(b) < 0
    if f(a) * f(b) >= 0:
        print("Selang tidak valid: f(a) * f(b) harus < 0.")
        return None

    langkah = []                 # menyimpan data setiap iterasi
    c_lama, galat = None, None
    for r in range(MAKS_ITER):
        c = (a + b) / 2                      # titik tengah selang [a, b]
        fa, fc, fb = f(a), f(c), f(b)        # nilai fungsi di a, c, dan b
        langkah.append({"r": r, "a": a, "c": c, "b": b,
                        "fa": fa, "fc": fc, "fb": fb})

        if fa * fc < 0:                      # f(a)f(c) < 0 ? ya
            b = c                            #   selang baru [a, b] <- [a, c]
            ket = "[a, c]"
        else:                                #   tidak
            a = c                            #   selang baru [a, b] <- [c, b]
            ket = "[c, b]"

        lebar = abs(a - b)                   # lebar selang yang baru
        langkah[-1]["ket"] = ket             # catat selang baru
        langkah[-1]["lebar"] = lebar         # catat lebarnya

        if c_lama is not None:
            galat = abs((c - c_lama) / c)    # galat relatif hampiran akar
        c_lama = c                           # simpan c untuk iterasi berikutnya

        # Kondisi berhenti: berhenti bila salah satu kriteria terpenuhi
        if lebar < EPSILON1 or abs(fc) < EPSILON2 or (galat is not None and galat < DELTA):
            break
    return langkah

## Langkah 4: Fungsi Pembuat Kurva 📈

Untuk menggambar kurva yang mulus, kita butuh banyak titik, tidak cukup 21 titik dari tabel.
Fungsi `kurva(lo, hi)` membuat **301 titik** yang jaraknya rapat dari `lo` sampai `hi`, lalu menghitung f(x) di tiap titiknya.
Hasilnya dipakai untuk menggambar garis biru di kedua diagram.

Cell ini hanya mendefinisikan fungsi, jadi tidak ada output.

In [4]:
def kurva(lo, hi, n=300):
    """Titik-titik halus (300 titik) untuk menggambar kurva y = f(x)."""
    xs = [lo + i * (hi - lo) / n for i in range(n + 1)]
    return xs, [f(v) for v in xs]

## Langkah 5: Cari Selang yang Mengandung Akar 🎯

Sekarang kita pakai `cari_selang` dengan pengaturan di Langkah 1.
Cell berikutnya akan:
1. Mencari semua selang yang f-nya berbeda tanda.
2. Mengambil **selang pertama** sebagai selang kerja (`a0`, `b0`).
3. Mencetak f(a0) dan f(b0) sebagai bukti bahwa tandanya berbeda.

Jika ada beberapa selang, Anda boleh memilih yang lain dengan mengganti `daftar_selang[0]` menjadi `daftar_selang[1]`, dan seterusnya.

In [5]:
x_tab, fx_tab, daftar_selang = cari_selang(A_AWAL, B_AKHIR, H)

print("Selang yang dapat dipilih:")
for s in daftar_selang:
    print(f"  [{s[0]:.2f}, {s[1]:.2f}]")

a0, b0 = daftar_selang[0]      # pakai selang pertama yang ditemukan
print(f"\nSelang yang dipakai: [{a0:.2f}, {b0:.2f}]")
print(f"f({a0:.2f}) = {f(a0):.6f}  dan  f({b0:.2f}) = {f(b0):.6f}")

Selang yang dapat dipilih:
  [2.00, 2.10]

Selang yang dipakai: [2.00, 2.10]
f(2.00) = -1.000000  dan  f(2.10) = 0.061000


### 👀 Membaca Output

    Selang yang dapat dipilih:
      [2.00, 2.10]

    Selang yang dipakai: [2.00, 2.10]
    f(2.00) = -1.000000  dan  f(2.10) = 0.061000

- Hanya ada **satu selang**: [2,00 ; 2,10]. Kurva ini hanya memotong sumbu-X satu kali di rentang 1 sampai 3.
- f(2,00) bernilai **negatif** (−1) dan f(2,10) bernilai **positif** (+0,061). Tandanya berbeda, jadi syarat metode **terpenuhi** dan akarnya pasti ada di dalam selang ini.

## Langkah 6: Diagram 1, Grafik f(x) dan Selang Berakar 📊

Diagram ini adalah **grafik fungsi di bidang X-Y** (sesuai materi), untuk melihat di mana kurva memotong sumbu-X.

### Cara membaca
| Tampilan | Artinya |
|---|---|
| **Garis biru** | Kurva y = f(x) |
| **Titik oranye** | Titik-titik dari tabel (jarak h = 0,1) |
| **Garis hitam datar** | Sumbu-X (y = 0) |
| **Daerah hijau** | Selang [a, b] yang f-nya berbeda tanda, tempat akar berada |

### Cara memakai
- **Arahkan kursor** ke titik oranye untuk melihat nilai x dan f(x).
- **Drag** (tarik) untuk memperbesar suatu bagian, dan **klik dua kali** untuk mengembalikan tampilan.
- Ikon kamera di pojok kanan atas menyimpan diagram sebagai gambar.

In [6]:
xs, ys = kurva(A_AWAL, B_AKHIR)             # titik-titik kurva

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=xs, y=ys, mode="lines",              # kurva f(x), biru
                          line=dict(color="royalblue", width=2),
                          hoverinfo="skip"))
fig1.add_trace(go.Scatter(x=x_tab, y=fx_tab, mode="markers",     # titik tabel, oranye
                          marker=dict(size=6, color="darkorange"),
                          hovertemplate="x = %{x:.2f}<br>f(x) = %{y:.6f}<extra></extra>"))
fig1.add_hline(y=0, line_color="black", line_width=1)            # sumbu-X
fig1.add_vrect(x0=a0, x1=b0, fillcolor="green", opacity=0.3, line_width=0,
               annotation_text=f"akar di [{a0:.2f}, {b0:.2f}]",
               annotation_position="top left", annotation_font_size=11)
fig1.update_layout(
    title=dict(text="Grafik f(x) dan selang yang mengandung akar", font_size=13),
    xaxis_title="x", yaxis_title="f(x)", showlegend=False,
    width=LEBAR, height=TINGGI, margin=dict(l=55, r=20, t=50, b=45),
    font_size=11)
fig1.show()

### 👀 Apa yang terlihat di Diagram 1

- Kurva biru naik dari kiri bawah ke kanan atas. Ia **memotong garis hitam (sumbu-X) satu kali**, tepat di dalam daerah hijau.
- Titik oranye di sebelah kiri daerah hijau berada **di bawah** sumbu-X (f negatif), dan yang di sebelah kanannya berada **di atas** (f positif). Itu alasan selang [2,0 ; 2,1] dipilih.
- Daerah hijau sangat sempit (lebar 0,1), jadi akarnya sudah **terkurung rapat** sebelum iterasi dimulai.

## Langkah 7: Jalankan Metode Bagidua ▶️

Sekarang fungsi `bagidua` kita jalankan pada selang [2,00 ; 2,10].
Seluruh data tiap iterasi (a, c, b, f(a), f(c), f(b), selang baru, lebar) disimpan di variabel `langkah`,
lalu dipakai untuk menggambar Diagram 2.

Cell ini mencetak ringkasan: jumlah iterasi, serta nilai a, c, b di iterasi pertama dan terakhir.## Langkah 7: Jalankan Metode Bagidua ▶️

Sekarang fungsi `bagidua` kita jalankan pada selang [2,00 ; 2,10].
Seluruh data tiap iterasi (a, c, b, f(a), f(c), f(b), selang baru, lebar) disimpan di variabel `langkah`,
lalu dipakai untuk menggambar Diagram 2.

Cell ini mencetak ringkasan: jumlah iterasi, serta nilai a, c, b di iterasi pertama dan terakhir.

In [7]:
langkah = bagidua(a0, b0)       # jalankan metode bagidua pada selang yang dipilih

print(f"Jumlah iterasi: {len(langkah)}")
print(f"Iterasi pertama : a = {langkah[0]['a']:.6f}, c = {langkah[0]['c']:.6f}, b = {langkah[0]['b']:.6f}")
print(f"Iterasi terakhir: a = {langkah[-1]['a']:.6f}, c = {langkah[-1]['c']:.6f}, b = {langkah[-1]['b']:.6f}")

Jumlah iterasi: 14
Iterasi pertama : a = 2.000000, c = 2.050000, b = 2.100000
Iterasi terakhir: a = 2.094543, c = 2.094550, b = 2.094556


### 👀 Membaca Output

    Jumlah iterasi: 14
    Iterasi pertama : a = 2.000000, c = 2.050000, b = 2.100000
    Iterasi terakhir: a = 2.094543, c = 2.094550, b = 2.094556

- Iterasi pertama: selang masih [2,0 ; 2,1] dan titik tengahnya 2,05.
- Iterasi terakhir: a dan b hampir **sama** (selisih hanya 0,000013), dan c sudah berada di antaranya.
- **Kenapa 14 iterasi?** Lebar selang awal 0,1 dibelah dua terus. Setelah 13 kali lebarnya 0,1 / 2¹³ ≈ 0,0000122 (belum di bawah 0,00001), dan setelah 14 kali 0,1 / 2¹⁴ ≈ 0,0000061 (sudah di bawah 0,00001), maka berhenti.

## Langkah 8: Diagram 2, Proses Pembagian Selang 🎞️

Diagram ini meniru gambar di materi: titik **a, c₀, c₁, c₂, ... , b** pada kurva.
Bedanya, diagram ini **interaktif**, jadi Anda bisa melihat prosesnya iterasi demi iterasi.

### Cara memakai
1. Lihat **slider di bawah diagram**. Angka 0 sampai 13 adalah nomor iterasi.
2. **Geser slider** pelan-pelan dari 0 ke kanan dan perhatikan perubahan di diagram.
3. **Baca judul** di atas diagram. Isinya a, b, c, selang baru, dan lebarnya untuk iterasi yang sedang tampil.
4. Arahkan kursor ke titik mana pun untuk melihat nilai x dan f(x) lengkap.

### Cara membaca
| Tampilan | Artinya |
|---|---|
| **Titik merah (a)** | Ujung kiri selang |
| **Titik oranye (b)** | Ujung kanan selang |
| **Titik hijau (c₀, c₁, ...)** | Titik tengah tiap iterasi, **bertambah** setiap slider maju |
| **Garis putus-putus abu-abu** | Penghubung titik ke sumbu-X, untuk melihat posisi x-nya |
| **Garis merah tebal di sumbu-X** | Selang [a, b] yang sedang dipakai |

Pada diagram ini, grafik **tidak melompat** saat slider digeser karena rentang tampilannya dikunci.

In [8]:
# Rentang tampilan dibuat tetap supaya grafik tidak melompat saat slider digeser
lo_x, hi_x = a0 - 0.1 * (b0 - a0), b0 + 0.1 * (b0 - a0)
xs2, ys2 = kurva(lo_x, hi_x)
margin = 0.25 * (max(ys2) - min(ys2))

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=xs2, y=ys2, mode="lines", hoverinfo="skip",   # kurva f(x)
                          line=dict(color="royalblue", width=2)))
fig2.add_hline(y=0, line_color="black", line_width=1)                      # sumbu-X

# Tiap iterasi punya 4 objek grafik (hanya iterasi 0 yang terlihat di awal)
for i, s in enumerate(langkah):
    tampil = (i == 0)
    c_semua = [langkah[k]["c"] for k in range(i + 1)]      # c0, c1, ..., ci
    fc_semua = [langkah[k]["fc"] for k in range(i + 1)]
    nama_c = [f"c{k}" for k in range(i + 1)]
    label_c = [n if (k <= 3 or k == i) else ""             # label dikurangi agar tidak
               for k, n in enumerate(nama_c)]              # menumpuk (hover tetap lengkap)

    # (1) garis putus-putus dari titik ke sumbu-X
    gx, gy = [], []
    for xv, yv in [(s["a"], s["fa"]), (s["b"], s["fb"])] + list(zip(c_semua, fc_semua)):
        gx += [xv, xv, None]
        gy += [0, yv, None]
    fig2.add_trace(go.Scatter(x=gx, y=gy, mode="lines",
                              line=dict(color="gray", dash="dot", width=1),
                              hoverinfo="skip", visible=tampil))

    # (2) titik a (merah) dan b (oranye)
    fig2.add_trace(go.Scatter(
        x=[s["a"], s["b"]], y=[s["fa"], s["fb"]], mode="markers+text",
        text=["a", "b"], textposition=["bottom center", "top center"],
        marker=dict(size=9, color=["red", "orange"]),
        customdata=["a", "b"],
        hovertemplate="%{customdata}<br>x = %{x:.6f}<br>f(x) = %{y:.6f}<extra></extra>",
        visible=tampil))

    # (3) titik c0 ... ci (hijau)
    fig2.add_trace(go.Scatter(
        x=c_semua, y=fc_semua, mode="markers+text", text=label_c,
        textposition=["top center" if v > 0 else "bottom center" for v in fc_semua],
        marker=dict(size=7, color="green"), customdata=nama_c,
        hovertemplate="%{customdata}<br>x = %{x:.6f}<br>f(x) = %{y:.6f}<extra></extra>",
        visible=tampil))

    # (4) selang [a, b] sebagai garis merah tebal di sumbu-X
    fig2.add_trace(go.Scatter(x=[s["a"], s["b"]], y=[0, 0], mode="lines",
                              line=dict(color="red", width=4),
                              hoverinfo="skip", visible=tampil))

# Slider: tiap posisi hanya menampilkan 4 objek milik iterasi tersebut
langkah_slider, judul_awal = [], ""
for i, s in enumerate(langkah):
    mask = [True] + [False] * (4 * len(langkah))    # kurva selalu terlihat
    for k in range(4):
        mask[1 + 4 * i + k] = True
    judul = (f"r = {s['r']}:  a = {s['a']:.6f},  b = {s['b']:.6f},  c{s['r']} = {s['c']:.6f}"
             f"<br>selang baru {s['ket']},  lebarnya = {s['lebar']:.6f}")
    if i == 0:
        judul_awal = judul
    langkah_slider.append(dict(method="update", label=str(s["r"]),
                               args=[{"visible": mask}, {"title": {"text": judul}}]))

fig2.update_layout(
    title=dict(text=judul_awal, font_size=12),
    xaxis=dict(title="x", range=[lo_x, hi_x]),
    yaxis=dict(title="f(x)", range=[min(ys2) - margin, max(ys2) + margin]),
    showlegend=False, width=LEBAR, height=TINGGI + 60, font_size=11,
    margin=dict(l=55, r=20, t=65, b=75),
    sliders=[dict(active=0, currentvalue=dict(prefix="Iterasi r = ", font_size=11),
                  pad=dict(t=35), steps=langkah_slider)])
fig2.show()

### 👀 Apa yang terlihat di Diagram 2

**Slider di 0 (awal):**
- Titik **a** (merah) di x = 2,0 berada di bawah sumbu-X, dan titik **b** (oranye) di x = 2,1 berada sedikit di atasnya.
- Titik **c₀** (hijau) di x = 2,05 berada di tengah, masih di bawah sumbu-X.
- Karena c₀ sama tanda dengan a, akar ada di kanan. Judul menulis *selang baru [c, b]*.

**Slider di 1 sampai 3:** titik a **bergeser ke kanan** mengikuti c sebelumnya, sementara b tetap. Titik hijau c₀, c₁, c₂, ... bertambah dan **mendekat ke sumbu-X**.

**Slider di 4:** c₄ berada **di atas** sumbu-X, jadi tandanya berubah. Sekarang **b yang bergeser** ke c₄ (judul menulis *[a, c]*).

**Slider di 13 (akhir):** semua titik hijau c₀ sampai c₁₃ menumpuk rapat di sekitar x = 2,0945. Garis merah tebal di sumbu-X hampir hanya berupa titik. Selang sudah sempit sekali, jadi akarnya tertangkap.

💡 Kalau titiknya terlalu rapat, **drag** untuk memperbesar tampilan, atau arahkan kursor untuk melihat nilai lengkapnya.

## Langkah 9: Hasil Akhir ✅

Cell terakhir mengambil data iterasi paling akhir, lalu mencetak:
- **Hampiran akar** x, yaitu nilai c di iterasi terakhir.
- **f(x)**, yang seharusnya sangat dekat dengan 0.
- **Jumlah iterasi** dan **lebar selang** terakhir.
- **Pemeriksaan:** akar dimasukkan kembali ke fungsi untuk melihat apakah hasilnya mendekati 0.

In [9]:
akhir = langkah[-1]            # data iterasi terakhir

print(f"Jadi, hampiran akarnya adalah x = {akhir['c']:.6f}")
print(f"f(x)           = {akhir['fc']:.6e}")
print(f"Jumlah iterasi = {len(langkah)}")
print(f"Lebar selang   = {akhir['lebar']:.6f}")

# Pemeriksaan: masukkan akar ke fungsi, hasilnya harus mendekati 0
print(f"\nPemeriksaan: f({akhir['c']:.6f}) = {f(akhir['c']):.6f}  (mendekati 0)")

Jadi, hampiran akarnya adalah x = 2.094550
f(x)           = -2.144105e-05
Jumlah iterasi = 14
Lebar selang   = 0.000006

Pemeriksaan: f(2.094550) = -0.000021  (mendekati 0)


### 👀 Membaca Hasil Akhir

    Jadi, hampiran akarnya adalah x = 2.094550
    f(x)           = -2.144105e-05
    Jumlah iterasi = 14
    Lebar selang   = 0.000006

    Pemeriksaan: f(2.094550) = -0.000021  (mendekati 0)

- **Akar** hampirannya adalah **x ≈ 2,094550**.
- **f(x) = −2,14 × 10⁻⁵** artinya **−0,0000214**. Itu sangat kecil, hampir nol.
- Hasilnya **tidak tepat 0** karena metode bagidua memberi *hampiran*. Ia berhenti saat selang cukup sempit (lebar < 0,00001), bukan saat f(x) tepat nol. Selisihnya dengan akar sebenarnya hanya sekitar 0,0000015.
- **Pemeriksaan:** memasukkan x = 2,094550 ke x³ − 2x − 5 menghasilkan sekitar −0,000021, dekat sekali dengan 0. Jadi jawabannya **benar**.

### Kesimpulan
Metode bagidua selalu berhasil menemukan akar jika selangnya benar (f(a) dan f(b) berbeda tanda), karena selang yang berisi akar terus dibelah dua dan **tidak pernah kehilangan akarnya**. Kelemahannya, ia **lambat**: tiap iterasi hanya menyempitkan selang separuhnya.### 👀 Membaca Hasil Akhir

    Jadi, hampiran akarnya adalah x = 2.094550
    f(x)           = -2.144105e-05
    Jumlah iterasi = 14
    Lebar selang   = 0.000006

    Pemeriksaan: f(2.094550) = -0.000021  (mendekati 0)

- **Akar** hampirannya adalah **x ≈ 2,094550**.
- **f(x) = −2,14 × 10⁻⁵** artinya **−0,0000214**. Itu sangat kecil, hampir nol.
- Hasilnya **tidak tepat 0** karena metode bagidua memberi *hampiran*. Ia berhenti saat selang cukup sempit (lebar < 0,00001), bukan saat f(x) tepat nol. Selisihnya dengan akar sebenarnya hanya sekitar 0,0000015.
- **Pemeriksaan:** memasukkan x = 2,094550 ke x³ − 2x − 5 menghasilkan sekitar −0,000021, dekat sekali dengan 0. Jadi jawabannya **benar**.

### Kesimpulan
Metode bagidua selalu berhasil menemukan akar jika selangnya benar (f(a) dan f(b) berbeda tanda), karena selang yang berisi akar terus dibelah dua dan **tidak pernah kehilangan akarnya**. Kelemahannya, ia **lambat**: tiap iterasi hanya menyempitkan selang separuhnya.